In [1]:
import sqlite3
import pandas as pd

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"

con = sqlite3.connect(DB_PATH)

# 1. lab_clean のカラム名確認
print("=== lab_clean columns ===")
print(pd.read_sql("PRAGMA table_info(lab_clean)", con))

# 2. tolvaptan_study のカラム名確認
print("\n=== tolvaptan_study columns ===")
print(pd.read_sql("PRAGMA table_info(tolvaptan_study)", con))

# 3. lab_clean の item_code サンプル（実際にどんな値が入っているか）
print("\n=== lab_clean item_code samples ===")
print(pd.read_sql("SELECT DISTINCT item_code FROM lab_clean LIMIT 20", con))

# 4. 対象 item_code が存在するか直接確認
print("\n=== target item_code count ===")
print(pd.read_sql("""
    SELECT item_code, COUNT(*) as cnt 
    FROM lab_clean 
    WHERE item_code IN ('0063','0074','0087','0093','0099','2550','2551')
    GROUP BY item_code
""", con))

# 5. tolvaptan_study の Patient_ID と lab_clean の Patient_ID が一致するか
print("\n=== ID overlap check ===")
print(pd.read_sql("""
    SELECT COUNT(DISTINCT l.Patient_ID) as matched_patients
    FROM lab_clean l
    INNER JOIN tolvaptan_study t ON l.Patient_ID = t.Patient_ID
""", con))

con.close()

=== lab_clean columns ===
    cid                name       type  notnull dflt_value  pk
0     0              raw_id    INTEGER        0       None   0
1     1          Patient_ID       TEXT        0       None   0
2     2         sample_date       TEXT        0       None   0
3     3           item_name       TEXT        0       None   0
4     4           item_code       TEXT        0       None   0
5     5           value_raw       TEXT        0       None   0
6     6                unit       TEXT        0       None   0
7     7             hl_flag       TEXT        0       None   0
8     8             ref_low       TEXT        0       None   0
9     9            ref_high       TEXT        0       None   0
10   10             comment       TEXT        0       None   0
11   11         lab_company       TEXT        0       None   0
12   12            facility       TEXT        0       None   0
13   13         source_file       TEXT        0       None   0
14   14         imported_at  

In [2]:
import sqlite3
import pandas as pd

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"
ITEM_CODES = ["0063", "0074", "0087", "0093", "0099", "2550", "2551"]
OUTPUT_CSV = "tolvaptan_lab_wide.csv"

con = sqlite3.connect(DB_PATH)

# ── 1. ind_date（患者ごとに最初の1件）──
sql_ind = """
    SELECT CAST(Patient_ID AS TEXT) AS Patient_ID, MIN(ind_date) AS ind_date
    FROM tolvaptan_study
    GROUP BY Patient_ID
"""
df_ind = pd.read_sql(sql_ind, con)
df_ind["ind_date"] = pd.to_datetime(df_ind["ind_date"])

# ── 2. lab_clean（Patient_ID を TEXT に統一）──
placeholders = ",".join("?" * len(ITEM_CODES))
sql_lab = f"""
    SELECT CAST(Patient_ID AS TEXT) AS Patient_ID,
           item_code, item_name, value_raw, sample_date_parsed AS sample_date
    FROM lab_clean
    WHERE item_code IN ({placeholders})
      AND sample_date_parsed IS NOT NULL
"""
df_lab = pd.read_sql(sql_lab, con, params=ITEM_CODES)
df_lab["sample_date"] = pd.to_datetime(df_lab["sample_date"])

con.close()

# ── 3. 患者 × item_code ごとに選択 ──
records = []

for _, row in df_ind.iterrows():
    pid = row["Patient_ID"]
    ind = row["ind_date"]

    labs = df_lab[df_lab["Patient_ID"] == pid].copy()

    for code in ITEM_CODES:
        subset = labs[labs["item_code"] == code].copy()

        before = subset[subset["sample_date"] <= ind].sort_values("sample_date", ascending=False).head(1)
        after  = subset[subset["sample_date"] >  ind].sort_values("sample_date", ascending=True).head(4)
        selected = pd.concat([before, after], ignore_index=True)

        iname = selected["item_name"].iloc[0] if not selected.empty else code

        rec = {"Patient_ID": pid, "item_code": code, "item_name": iname, "ind_date": ind.date()}
        for i in range(5):
            rec[f"sample_date_{i+1}"] = selected.iloc[i]["sample_date"].date() if i < len(selected) else None
            rec[f"value_raw_{i+1}"]   = selected.iloc[i]["value_raw"]           if i < len(selected) else None

        records.append(rec)

df_out = pd.DataFrame(records)

cols = ["Patient_ID", "item_code", "item_name", "ind_date"]
for i in range(1, 6):
    cols += [f"sample_date_{i}", f"value_raw_{i}"]

df_out = df_out[cols]
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Done: {len(df_out)} rows → {OUTPUT_CSV}")
df_out.head(14)

Done: 1323 rows → tolvaptan_lab_wide.csv


,Patient_ID,item_code,item_name,ind_date,sample_date_1,value_raw_1,sample_date_2,value_raw_2,sample_date_3,value_raw_3,sample_date_4,value_raw_4,sample_date_5,value_raw_5
0,150004,0063,尿素窒素,2017-12-06,2017-12-01,47.2,2017-12-07,43.1,2017-12-09,41.8,2017-12-11,36.9,2018-01-26,21.3
1,150004,0074,ＣＲＥ,2017-12-06,2017-12-01,1.56,2017-12-07,1.59,2017-12-09,1.66,2017-12-11,1.53,2018-01-26,0.99
2,150004,0087,ナトリウム,2017-12-06,2017-12-01,140,2017-12-07,142,2017-12-09,143,2017-12-11,142,2018-01-26,140
3,150004,0093,カリウム,2017-12-06,2017-12-01,4.4,2017-12-07,5.1,2017-12-09,4.5,2017-12-11,5.6,2018-01-26,5.7
4,150004,0099,クロール,2017-12-06,2017-12-01,98,2017-12-07,100,2017-12-09,100,2017-12-11,103,2018-01-26,100
5,150004,2550,ＢＮＰ,2017-12-06,2017-12-01,259.1,2018-01-26,267.0,2018-02-09,301.4,2018-03-23,159.6,2018-06-15,112.2
6,150004,2551,プロＢＮＰ,2017-12-06,2017-12-01,2538,2018-01-26,1886,2018-02-09,1855,2018-03-23,824,2018-06-15,495
7,150009,0063,尿素窒素,2016-12-01,2016-11-16,22.5,2016-12-02,25.7,2016-12-05,26.3,2016-12-09,35.9,2016-12-12,51.2
8,150009,0074,ＣＲＥ,2016-12-01,2016-11-16,1.59,2016-12-02,1.56,2016-12-05,1.52,2016-12-09,1.67,2016-12-12,1.7
9,150009,0087,ナトリウム,2016-12-01,2016-11-16,142,2016-12-02,140,2016-12-05,141,2016-12-09,140,2016-12-12,141


In [3]:
import pandas as pd

CSV_PATH = "/Users/muna/Hana_research/data/processed/tolvaptan_lab_wide.csv"
df = pd.read_csv(CSV_PATH)

# item_code の実際の値を確認
print(df["item_code"].unique())
print(df["item_code"].dtype)

[  63   74   87   93   99 2550 2551]
int64


In [4]:
import pandas as pd

CSV_PATH = "/Users/muna/Hana_research/data/processed/tolvaptan_lab_wide.csv"

# item_code を文字列で読み込む
df = pd.read_csv(CSV_PATH, dtype={"item_code": str})

df87 = df[df["item_code"] == "0087"].copy()
total = len(df87)

print(f"item_code 0087 - 対象患者数: {total}人\n")
print(f"{'列':<15} {'測定あり':>8} {'測定なし':>8} {'測定率':>8}")
print("-" * 45)

for i in range(1, 6):
    col = f"value_raw_{i}"
    n_valid = df87[col].notna().sum()
    n_null  = total - n_valid
    rate    = n_valid / total * 100
    print(f"sample_date_{i}  {n_valid:>6}人  {n_null:>6}人  {rate:>6.1f}%")

item_code 0087 - 対象患者数: 189人

列                   測定あり     測定なし      測定率
---------------------------------------------
sample_date_1     189人       0人   100.0%
sample_date_2     182人       7人    96.3%
sample_date_3     163人      26人    86.2%
sample_date_4     148人      41人    78.3%
sample_date_5     128人      61人    67.7%


In [5]:
import pandas as pd

CSV_PATH = "/Users/muna/Hana_research/data/processed/tolvaptan_lab_wide.csv"

df = pd.read_csv(CSV_PATH, dtype={"item_code": str})
df87 = df[df["item_code"] == "0087"].copy()

# ind_date をdatetime化
df87["ind_date"] = pd.to_datetime(df87["ind_date"])

# 各sample_dateをdatetime化 & 差分計算
for i in range(1, 6):
    sd = f"sample_date_{i}"
    dd = f"days_diff_{i}"
    df87[sd] = pd.to_datetime(df87[sd])
    df87[dd] = (df87[sd] - df87["ind_date"]).dt.days

# 出力列を整理（1患者1行・wide形式）
cols = ["Patient_ID", "ind_date"]
for i in range(1, 6):
    cols += [f"sample_date_{i}", f"days_diff_{i}"]

df_out = df87[cols].reset_index(drop=True)

print(f"患者数: {len(df_out)}人")
df_out

患者数: 189人


,Patient_ID,ind_date,sample_date_1,days_diff_1,sample_date_2,days_diff_2,sample_date_3,days_diff_3,sample_date_4,days_diff_4,sample_date_5,days_diff_5
0,150004,2017-12-06,2017-12-01,-5,2017-12-07,1.0,2017-12-09,3.0,2017-12-11,5.0,2018-01-26,51.0
1,150009,2016-12-01,2016-11-16,-15,2016-12-02,1.0,2016-12-05,4.0,2016-12-09,8.0,2016-12-12,11.0
2,150027,2016-09-30,2016-08-03,-58,2016-10-01,1.0,2016-10-03,3.0,2016-10-11,11.0,NaT,NaN
3,150041,2016-05-10,2016-05-02,-8,2016-05-12,2.0,2016-05-16,6.0,2016-06-07,28.0,2016-07-12,63.0
4,150061,2016-07-19,2016-07-05,-14,2016-07-26,7.0,NaT,NaN,NaT,NaN,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
184,240181,2026-04-01,2026-03-30,-2,2026-04-06,5.0,2026-04-07,6.0,NaT,NaN,NaT,NaN
185,240256,2024-10-01,2024-10-01,0,2024-10-08,7.0,2024-10-15,14.0,2024-11-19,49.0,2024-12-11,71.0
186,240277,2024-10-28,2024-10-11,-17,2024-10-30,2.0,2024-11-01,4.0,2024-11-05,8.0,2025-02-10,105.0
187,240287,2024-11-07,2024-11-06,-1,2024-11-11,4.0,2024-11-14,7.0,2024-11-20,13.0,2024-11-29,22.0
